In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'rdflib', 'owlrl'], check=False)
print('Dependencies ready.')

In [ ]:
import json, re, time
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from rdflib import Graph, RDF, RDFS, OWL, URIRef, Literal, Namespace, BNode
from owlrl import DeductiveClosure, OWLRL_Semantics
print('Imports loaded.')

def save_dual_format(graph, ttl_path, owl_path):
    """Save graph as both Turtle (.ttl) and RDF/XML (.owl)."""
    graph.serialize(destination=str(ttl_path), format='turtle')
    n_full = len(graph)
    xml_safe = Graph()
    for prefix, ns in graph.namespaces():
        xml_safe.bind(prefix, ns)
    dropped = 0
    for s, p, o in graph:
        if isinstance(s, Literal):
            dropped += 1
            continue
        xml_safe.add((s, p, o))
    xml_safe.serialize(destination=str(owl_path), format='xml')
    return n_full, len(xml_safe), dropped

def clean_reasoner_noise(graph):
    """Remove OWL RL artifacts and OWL 2 DL violations."""
    cleaned = Graph()
    for prefix, ns in graph.namespaces():
        cleaned.bind(prefix, ns)
    removed = Counter()
    
    ERROR_MSG = URIRef('http://www.daml.org/2002/03/agents/agent-ont#ErrorMessage')
    
    for s, p, o in graph:
        # Remove sameAs involving blank nodes (OWL 2 DL violation)
        if p == OWL.sameAs and (isinstance(s, BNode) or isinstance(o, BNode)):
            removed['sameAs_with_blank_node'] += 1
            continue
        
        # Remove ErrorMessage references (stray import)
        if s == ERROR_MSG or o == ERROR_MSG:
            removed['error_message_ref'] += 1
            continue
        
        # Standard OWL RL noise
        if s == OWL.Nothing and p == RDFS.subClassOf:
            removed['nothing_subclassof'] += 1
            continue
        if isinstance(s, Literal):
            removed['literal_subject'] += 1
            continue
        if p == RDFS.subClassOf and s == o:
            removed['reflexive_subclass'] += 1
            continue
        if p == RDFS.subClassOf and o == OWL.Thing:
            removed['subclass_of_thing'] += 1
            continue
        if p == OWL.equivalentClass and s == o:
            removed['reflexive_equivalentclass'] += 1
            continue
        if p == OWL.sameAs and s == o and isinstance(s, URIRef):
            removed['reflexive_sameas_uri'] += 1
            continue
        
        cleaned.add((s, p, o))
    
    return cleaned, removed

print('Helper functions defined.')

In [ ]:
BASE_DIR = Path('/Users/umair/Synthesising Regulatory Ontologies')

# Inputs
CCO_PATH        = BASE_DIR / '1 - Foundation Layer' / 'CCO.ttl'
PER_JUR_DIR     = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'per_jurisdiction_merged'
STAGE5B_TTL     = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'stage5b_alignment' / 'cross_jurisdiction_alignments.ttl'
STAGE5C_TTL     = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'stage5c_shared_superclass' / 'stage5c_shared_superclasses.ttl'

# Output
OUTPUT_DIR             = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'stage5d_merged_reasoned'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Audit trail
ASSERTED_BASELINE_TTL  = OUTPUT_DIR / 'unified_asserted.ttl'
ASSERTED_PATCHED_TTL   = OUTPUT_DIR / 'unified_asserted_patched.ttl'
REASONED_RAW_TTL       = OUTPUT_DIR / 'unified_reasoned_raw.ttl'

# PRIMARY DELIVERABLE
UNIFIED_TTL            = OUTPUT_DIR / 'unified_ontology.ttl'
UNIFIED_OWL            = OUTPUT_DIR / 'unified_ontology.owl'

# Report
REPORT_PATH            = OUTPUT_DIR / 'stage5d_reasoning_report.json'

JURISDICTIONS = ['gro-uk', 'gro-us', 'gro-ca', 'gro-au']

# CCO namespace URIs (used in patches & disjointness fixes)
SKOS_ALT      = URIRef('http://www.w3.org/2004/02/skos/core#altLabel')
PROV_DERIVED  = URIRef('http://www.w3.org/ns/prov#wasDerivedFrom')
DCTERMS       = Namespace('http://purl.org/dc/terms/')

CCO_NORM        = URIRef('https://www.w3id.org/cco/cco#Norm')
CCO_EXCEPTION   = URIRef('https://www.w3id.org/cco/cco#Exception')
CCO_OBLIGATION  = URIRef('https://www.w3id.org/cco/cco#Obligation')
CCO_PERMISSION  = URIRef('https://www.w3id.org/cco/cco#Permission')
CCO_PROHIBITION = URIRef('https://www.w3id.org/cco/cco#Prohibition')
CCO_RESOURCE    = URIRef('https://www.w3id.org/cco/cco#Resource')
CCO_AGENT       = URIRef('https://www.w3id.org/cco/cco#Agent')
CCO_RA_AGENT    = URIRef('https://www.w3id.org/cco/cco#RegulatoryAuthorityAgent')
CCO_ORG         = URIRef('https://www.w3id.org/cco/cco#Organisation')
CCO_PERSON      = URIRef('https://www.w3id.org/cco/cco#Person')

# Persistent ontology IRI
ONTOLOGY_IRI         = URIRef('https://w3id.org/cco-gro/onto')
ONTOLOGY_VERSION_IRI = URIRef('https://w3id.org/cco-gro/onto/v1.0')
ONTOLOGY_TITLE       = 'Cross-Jurisdiction Regulatory Ontology for Education Funding'

# Verify all inputs exist
assert CCO_PATH.exists(),    f'CCO not found: {CCO_PATH}'
assert PER_JUR_DIR.exists(), f'Per-jur dir not found: {PER_JUR_DIR}'
assert STAGE5B_TTL.exists(), f'Stage 5B TTL not found: {STAGE5B_TTL}'
assert STAGE5C_TTL.exists(), f'Stage 5C TTL not found: {STAGE5C_TTL}'

print(f'CCO:           {CCO_PATH.name}')
print(f'Per-jur dir:   {PER_JUR_DIR.name}')
print(f'Stage 5B:      {STAGE5B_TTL.name}')
print(f'Stage 5C:      {STAGE5C_TTL.name}')
print(f'Output dir:    {OUTPUT_DIR.name}')
print(f'\nPrimary deliverable: {UNIFIED_TTL.name} + {UNIFIED_OWL.name}')

In [ ]:
g = Graph()
load_log = []

# CCO foundation
print('Loading CCO ontology...')
n_before = len(g)
g.parse(str(CCO_PATH), format='turtle')
load_log.append({'source': 'CCO.ttl', 'triples_added': len(g) - n_before})
print(f'  + {len(g) - n_before} triples')

# Per-jurisdiction TTLs
print('\nLoading per-jurisdiction TTLs...')
for jur in JURISDICTIONS:
    path = PER_JUR_DIR / f'merged_{jur}.ttl'
    if not path.exists():
        print(f'  WARNING: {path.name} not found')
        continue
    n_before = len(g)
    g.parse(str(path), format='turtle')
    delta = len(g) - n_before
    load_log.append({'source': f'merged_{jur}.ttl', 'triples_added': delta})
    print(f'  merged_{jur}.ttl: +{delta} triples')

# Stage 5B alignments
print('\nLoading Stage 5B alignments...')
n_before = len(g)
g.parse(str(STAGE5B_TTL), format='turtle')
delta = len(g) - n_before
load_log.append({'source': 'cross_jurisdiction_alignments.ttl', 'triples_added': delta})
print(f'  Stage 5B: +{delta} triples')

# Stage 5C shared superclasses
print('\nLoading Stage 5C shared superclasses...')
n_before = len(g)
g.parse(str(STAGE5C_TTL), format='turtle')
delta = len(g) - n_before
load_log.append({'source': 'stage5c_shared_superclasses.ttl', 'triples_added': delta})
print(f'  Stage 5C: +{delta} triples')

asserted_pre_patch = len(g)
print(f'\nTotal merged graph (pre-patch): {asserted_pre_patch} triples')

g.serialize(destination=str(ASSERTED_BASELINE_TTL), format='turtle')
print(f'Asserted baseline saved: {ASSERTED_BASELINE_TTL.name}')

In [ ]:
patch_log = []

def camel_to_words(name):
    name = re.sub(r'([a-z])([A-Z])', r'\1 \2', name)
    name = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', name)
    return name

# ===== Fix 1: Pronoun label resolution =====
print('Fix 1: Pronoun label resolution')
PRONOUN_LABELS = {'you', 'they', 'we', 'i', 'he', 'she', 'it'}

pronoun_targets = []
for s, p, o in g.triples((None, RDFS.label, None)):
    if isinstance(o, Literal) and str(o).strip().lower() in PRONOUN_LABELS:
        pronoun_targets.append((s, o))

fix1_count = 0
for target, old_label in pronoun_targets:
    types = list(g.objects(target, RDF.type))
    gro_types = [t for t in types if isinstance(t, URIRef) and 'cco-gro/onto' in str(t)]
    cco_types = [t for t in types if isinstance(t, URIRef) and 'cco/cco' in str(t)]
    
    replacement = None
    if gro_types:
        type_local = str(gro_types[0]).split('#')[-1]
        replacement = camel_to_words(type_local).replace('Role', '').strip()
        if not replacement:
            replacement = camel_to_words(type_local)
    elif cco_types:
        type_local = str(cco_types[0]).split('#')[-1]
        replacement = camel_to_words(type_local)
    
    if replacement:
        g.remove((target, RDFS.label, old_label))
        g.add((target, RDFS.label, Literal(replacement)))
        g.add((target, SKOS_ALT, old_label))
        patch_log.append({
            'fix_id': 'fix_1_pronoun', 'target': str(target),
            'from': str(old_label), 'to': replacement,
        })
        fix1_count += 1
print(f'  Applied: {fix1_count} pronoun resolutions')

# ===== Fix 2: Fallback labels for unlabeled Norm individuals =====
print('\nFix 2: Fallback labels for unlabeled Norms')
norm_individuals = sorted({
    s for s, p, o in g.triples((None, RDF.type, None))
    if isinstance(s, URIRef) and '_norm_' in str(s) and 'cco-gro/data#' in str(s)
}, key=str)

fix2_count = 0
for n in norm_individuals:
    if list(g.objects(n, RDFS.label)):
        continue
    types = [t for t in g.objects(n, RDF.type) if isinstance(t, URIRef)]
    gro_types = [t for t in types if 'cco-gro/onto' in str(t)]
    
    if gro_types:
        type_local = str(gro_types[0]).split('#')[-1]
        type_label = camel_to_words(type_local)
        source = list(g.objects(n, PROV_DERIVED))
        if source:
            src_local = str(source[0]).split('#')[-1].replace('_', '-')
            fallback_label = f'{type_label} ({src_local})'
        else:
            fallback_label = type_label
    else:
        fallback_label = 'Unlabeled Norm'
    
    g.add((n, RDFS.label, Literal(fallback_label)))
    patch_log.append({'fix_id': 'fix_2_fallback', 'target': str(n), 'to': fallback_label})
    fix2_count += 1

print(f'  Applied: {fix2_count} fallback labels')

# ===== Fix 3: Exception / Norm disjointness conflicts =====
print('\nFix 3: Exception vs Norm disjointness')
exc_norm_conflicts = []
for s in g.subjects(RDF.type, CCO_EXCEPTION):
    for wrong_type in [CCO_NORM, CCO_OBLIGATION, CCO_PERMISSION, CCO_PROHIBITION]:
        if (s, RDF.type, wrong_type) in g:
            exc_norm_conflicts.append((s, wrong_type))

for s, wt in exc_norm_conflicts:
    g.remove((s, RDF.type, wt))
    patch_log.append({'fix_id': 'fix_3_exception_norm', 'target': str(s), 'removed_type': str(wt)})

print(f'  Applied: removed {len(exc_norm_conflicts)} wrong types from Exceptions')

# ===== Fix 4: Norm vs Agent/Resource/Organisation/Person conflicts =====
print('\nFix 4: Norm + non-Norm type conflicts')
norm_individuals = set(g.subjects(RDF.type, CCO_NORM))
wrong_types_for_norm = [CCO_AGENT, CCO_RA_AGENT, CCO_RESOURCE, CCO_ORG, CCO_PERSON]

norm_conflicts = []
for s in norm_individuals:
    for wt in wrong_types_for_norm:
        if (s, RDF.type, wt) in g:
            norm_conflicts.append((s, wt))

for s, wt in norm_conflicts:
    g.remove((s, RDF.type, wt))
    patch_log.append({'fix_id': 'fix_4_norm_conflict', 'target': str(s), 'removed_type': str(wt)})

print(f'  Applied: removed {len(norm_conflicts)} wrong types from Norms')

# ===== Fix 5: Person + Organisation conflicts (disjoint) =====
print('\nFix 5: Person + Organisation disjointness')
po_conflicts = []
for s in g.subjects(RDF.type, CCO_PERSON):
    if (s, RDF.type, CCO_ORG) in g:
        po_conflicts.append(s)

for s in po_conflicts:
    g.remove((s, RDF.type, CCO_ORG))
    patch_log.append({'fix_id': 'fix_5_person_org', 'target': str(s)})

print(f'  Applied: {len(po_conflicts)} Person/Org conflicts')

# ===== Fix 6: Exception individuals with Norm-domain properties =====
print('\nFix 6: Exception individuals using Norm-domain properties')
norm_domain_props = [p for s, _, o in g.triples((None, RDFS.domain, CCO_NORM)) for p in [s]]
print(f'  Norm-domain properties detected: {len(norm_domain_props)}')

prop_removals = 0
exceptions_list = list(g.subjects(RDF.type, CCO_EXCEPTION))
for exc in exceptions_list:
    for prop in norm_domain_props:
        triples_to_remove = list(g.triples((exc, prop, None)))
        for tr in triples_to_remove:
            g.remove(tr)
            prop_removals += 1
            patch_log.append({'fix_id': 'fix_6_exc_norm_prop',
                              'target': str(exc), 'removed_prop': str(prop)})

print(f'  Applied: {prop_removals} property triples removed')

# Summary
asserted_patched = len(g)
print(f'\nTotal patches applied: {len(patch_log)}')
print(f'Triples after patches: {asserted_patched} (was {asserted_pre_patch})')

g.serialize(destination=str(ASSERTED_PATCHED_TTL), format='turtle')
print(f'Asserted patched saved: {ASSERTED_PATCHED_TTL.name}')

In [ ]:
print('Applying OWL RL reasoning...')

asserted_set = set(g)
start_time = time.time()

try:
    DeductiveClosure(OWLRL_Semantics).expand(g)
    reasoning_status = 'completed'
    reasoning_error = None
    elapsed = time.time() - start_time
    print(f'  Reasoning completed in {elapsed/60:.1f} minutes')
except Exception as e:
    reasoning_status = 'failed'
    reasoning_error = str(e)[:300]
    print(f'  REASONING FAILED: {reasoning_error}')

post_reasoning_count = len(g)
inferred_count = post_reasoning_count - asserted_patched

print(f'  Asserted (patched):     {asserted_patched}')
print(f'  Post-reasoning:         {post_reasoning_count}')
print(f'  Inferred triples added: {inferred_count}')

g.serialize(destination=str(REASONED_RAW_TTL), format='turtle')
print(f'Raw reasoned saved: {REASONED_RAW_TTL.name}')

In [ ]:
inferred_set = set(g) - asserted_set
predicate_counts = Counter()
for s, p, o in inferred_set:
    predicate_counts[str(p)] += 1

print('Top inferred predicates:')
for pred, count in predicate_counts.most_common(10):
    pred_short = pred.split('#')[-1] if '#' in pred else pred.split('/')[-1]
    print(f'  {pred_short:30s}: {count}')

subclass_inferred = [(s, o) for s, p, o in inferred_set if p == RDFS.subClassOf]
equiv_inferred = [(s, o) for s, p, o in inferred_set if p == OWL.equivalentClass]
type_inferred = [(s, o) for s, p, o in inferred_set if p == RDF.type]

# Cross-jurisdiction inferences
JUR_NSS = {
    'gro-uk': 'https://w3id.org/cco-gro/onto/uk#',
    'gro-us': 'https://w3id.org/cco-gro/onto/us#',
    'gro-ca': 'https://w3id.org/cco-gro/onto/ca#',
    'gro-au': 'https://w3id.org/cco-gro/onto/au#',
}

def jur_of(uri):
    s = str(uri)
    for pfx, ns in JUR_NSS.items():
        if s.startswith(ns):
            return pfx
    return 'other'

cross_jur_sub = []
for s, o in subclass_inferred:
    js, jo = jur_of(s), jur_of(o)
    if js in JUR_NSS and jo in JUR_NSS and js != jo:
        cross_jur_sub.append((s, o, js, jo))

cross_jur_eq = []
for s, o in equiv_inferred:
    js, jo = jur_of(s), jur_of(o)
    if js in JUR_NSS and jo in JUR_NSS and js != jo:
        cross_jur_eq.append((s, o, js, jo))

print(f'\nKey inference categories:')
print(f'  rdfs:subClassOf:     {len(subclass_inferred)}')
print(f'  owl:equivalentClass: {len(equiv_inferred)}')
print(f'  rdf:type:            {len(type_inferred)}')
print(f'\nCross-jurisdiction inferences:')
print(f'  rdfs:subClassOf:     {len(cross_jur_sub)}')
print(f'  owl:equivalentClass: {len(cross_jur_eq)}')

In [ ]:
inconsistencies = []

# Check 1: owl:Nothing instances
nothing_inst = [x for x in g.subjects(RDF.type, OWL.Nothing) if x != OWL.Nothing]
if nothing_inst:
    inconsistencies.append({'type': 'nothing_instances', 'count': len(nothing_inst)})

# Check 2: subClassOf cycles (excluding legit OWL entailments)
equiv_pairs = set()
for s, o in g.subject_objects(OWL.equivalentClass):
    if isinstance(s, URIRef) and isinstance(o, URIRef):
        equiv_pairs.add((s, o))
        equiv_pairs.add((o, s))

class_subclass = defaultdict(set)
for s, o in g.subject_objects(RDFS.subClassOf):
    if not (isinstance(s, URIRef) and isinstance(o, URIRef)):
        continue
    if s == o or s == OWL.Nothing or o == OWL.Nothing:
        continue
    if (s, o) in equiv_pairs:
        continue
    class_subclass[s].add(o)

def has_cycle(start, adj, visited=None, stack=None):
    visited = visited if visited is not None else set()
    stack = stack if stack is not None else set()
    if start in stack: return True
    if start in visited: return False
    visited.add(start)
    stack.add(start)
    for nxt in adj.get(start, ()):
        if has_cycle(nxt, adj, visited, stack):
            return True
    stack.discard(start)
    return False

cycle_nodes = []
checked = set()
for node in list(class_subclass):
    if node in checked: continue
    visited = set()
    if has_cycle(node, class_subclass, visited, set()):
        cycle_nodes.append(str(node))
    checked.update(visited)

if cycle_nodes:
    inconsistencies.append({'type': 'subclass_cycles', 'count': len(cycle_nodes)})

# Check 3: sameAs/differentFrom conflicts
sameas = set(g.subject_objects(OWL.sameAs))
diff = set(g.subject_objects(OWL.differentFrom))
conflicts = sameas & diff
if conflicts:
    inconsistencies.append({'type': 'sameas_diff_conflict', 'count': len(conflicts)})

if inconsistencies:
    print('INCONSISTENCIES:')
    for inc in inconsistencies:
        print(f'  {inc["type"]}: {inc["count"]}')
else:
    print('No inconsistencies detected by OWL RL.')

In [ ]:
print('Cleaning reasoner noise + OWL 2 violations...')
g_clean, removed = clean_reasoner_noise(g)

print(f'  Triples before: {len(g)}')
print(f'  Triples after:  {len(g_clean)}')
print(f'  Removed:        {sum(removed.values())}')
for reason, count in removed.most_common():
    print(f'    {reason}: {count}')

# Add ontology declaration
g_clean.bind('dcterms', DCTERMS)
g_clean.add((ONTOLOGY_IRI, RDF.type, OWL.Ontology))
g_clean.add((ONTOLOGY_IRI, OWL.versionIRI, ONTOLOGY_VERSION_IRI))
g_clean.add((ONTOLOGY_IRI, DCTERMS.title, Literal(ONTOLOGY_TITLE)))
g_clean.add((ONTOLOGY_IRI, DCTERMS.issued, Literal(datetime.now().date().isoformat())))
g_clean.add((ONTOLOGY_IRI, RDFS.comment, Literal(
    'Cross-jurisdiction regulatory ontology for education funding regulations. '
    'Extends a Core Compliance Ontology (CCO) foundation with four per-jurisdiction '
    'sub-vocabularies (UK, US, CA, AU). Cross-jurisdiction concepts aligned via the '
    'Functional Equivalence pattern and abstracted via the Shared Domain Superclass pattern.'
)))

n_ttl, n_owl, dropped_xml = save_dual_format(g_clean, UNIFIED_TTL, UNIFIED_OWL)
print(f'\nPRIMARY DELIVERABLE:')
print(f'  TTL: {UNIFIED_TTL.name} ({n_ttl} triples)')
print(f'  OWL: {UNIFIED_OWL.name} ({n_owl} triples)')
if dropped_xml:
    print(f'  Note: {dropped_xml} triples dropped from .owl (RDF/XML constraints)')

In [ ]:
report = {
    'metadata': {
        'stage': 'stage5d_merge_and_reason',
        'created_at': datetime.now().isoformat(),
        'ontology_iri': str(ONTOLOGY_IRI),
        'ontology_version': str(ONTOLOGY_VERSION_IRI),
    },
    'inputs': {
        'cco_path': str(CCO_PATH),
        'per_jurisdiction_dir': str(PER_JUR_DIR),
        'stage5b_ttl': str(STAGE5B_TTL),
        'stage5c_ttl': str(STAGE5C_TTL),
    },
    'load_log': load_log,
    'patches': {
        'total_applied': len(patch_log),
        'fix_1_pronoun': sum(1 for p in patch_log if p['fix_id'] == 'fix_1_pronoun'),
        'fix_2_fallback': sum(1 for p in patch_log if p['fix_id'] == 'fix_2_fallback'),
        'fix_3_exception_norm': sum(1 for p in patch_log if p['fix_id'] == 'fix_3_exception_norm'),
        'fix_4_norm_conflict': sum(1 for p in patch_log if p['fix_id'] == 'fix_4_norm_conflict'),
        'fix_5_person_org': sum(1 for p in patch_log if p['fix_id'] == 'fix_5_person_org'),
        'fix_6_exc_norm_prop': sum(1 for p in patch_log if p['fix_id'] == 'fix_6_exc_norm_prop'),
    },
    'reasoning': {
        'status': reasoning_status,
        'asserted_pre_patch': asserted_pre_patch,
        'asserted_patched': asserted_patched,
        'post_reasoning': post_reasoning_count,
        'inferred_added': inferred_count,
    },
    'inferences': {
        'subclass_inferred': len(subclass_inferred),
        'equivalentclass_inferred': len(equiv_inferred),
        'type_inferred': len(type_inferred),
        'cross_jurisdiction_subclass': len(cross_jur_sub),
        'cross_jurisdiction_equivalentclass': len(cross_jur_eq),
    },
    'inconsistencies': inconsistencies,
    'cleanup': {
        'triples_before': len(g),
        'triples_after': len(g_clean),
        'removed_count': sum(removed.values()),
        'removed_breakdown': dict(removed),
    },
    'final': {
        'unified_ttl_triples': n_ttl,
        'unified_owl_triples': n_owl,
    },
}

with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, default=str)
print(f'Report saved: {REPORT_PATH.name}')

# Final summary
print('\n' + '=' * 70)
print('STAGE 5D FINAL SUMMARY')
print('=' * 70)
print(f'\nInputs:')
for entry in load_log:
    print(f'  {entry["source"]:50s} +{entry["triples_added"]}')

print(f'\nPatches applied (total {len(patch_log)}):')
print(f'  Fix 1 (pronoun resolution):       {report["patches"]["fix_1_pronoun"]}')
print(f'  Fix 2 (fallback labels):          {report["patches"]["fix_2_fallback"]}')
print(f'  Fix 3 (Exception/Norm conflict):  {report["patches"]["fix_3_exception_norm"]}')
print(f'  Fix 4 (Norm type conflicts):      {report["patches"]["fix_4_norm_conflict"]}')
print(f'  Fix 5 (Person/Org disjoint):      {report["patches"]["fix_5_person_org"]}')
print(f'  Fix 6 (Exc/Norm-prop conflict):   {report["patches"]["fix_6_exc_norm_prop"]}')

print(f'\nPipeline progression:')
print(f'  Pre-patch:                  {asserted_pre_patch}')
print(f'  Patched:                    {asserted_patched}')
print(f'  Post-reasoning:             {post_reasoning_count}')
print(f'  After cleanup:              {n_ttl}')

print(f'\nCross-jurisdiction inferences:')
print(f'  rdfs:subClassOf:            {len(cross_jur_sub)}')
print(f'  owl:equivalentClass:        {len(cross_jur_eq)}')

print(f'\nInconsistencies: {len(inconsistencies)}')

print(f'\nOUTPUT FILES:')
print(f'  PRIMARY: {UNIFIED_TTL.name} ({n_ttl} triples)')
print(f'  PRIMARY: {UNIFIED_OWL.name} ({n_owl} triples)')
print(f'  Audit:   {ASSERTED_BASELINE_TTL.name}')
print(f'  Audit:   {ASSERTED_PATCHED_TTL.name}')
print(f'  Audit:   {REASONED_RAW_TTL.name}')
print(f'  Report:  {REPORT_PATH.name}')

print(f'\nDone. Open {UNIFIED_OWL.name} in Protege, run ELK or HermiT reasoner.')